# 猪咳嗽声智能监测模块 —— 完整流程演示

基于《猪咳嗽声智能监测模块》设计文档 + 付小朋(CNN-BiLSTM) + 王梦雨(预警系统)。

> 运行环境：Python 3.10+（TensorFlow / librosa / scikit-learn，见 `requirements/monitor.txt`）。
> 本 Notebook 在 `monitor/notebooks/` 目录下运行，路径自动指向项目根目录。

## 0. 环境与配置检查

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import config
print("采样率:", config.SAMPLE_RATE, "| 特征维数:", config.N_FEATURES, "| 固定帧数:", config.N_FRAMES)
print("输入形状:", config.INPUT_SHAPE)
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "CPU 模式")

In [ ]:
from pathlib import Path
n_cough = len(list(Path(config.RAW_DIR).glob("cough/*.wav")))
n_non = len(list(Path(config.RAW_DIR).glob("non_cough/*.wav")))
print(f"训练数据: 咳嗽 {n_cough} / 非咳嗽 {n_non} （来自 data/raw/monitor，整理见 scripts/10_build_retrain_dataset.py）")

## 2. 步骤1：信号预处理

四阶巴特沃斯带通(100~2000Hz) → 预加重(0.97) → 短时能量端点检测。下图对比处理前后波形。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.audio_io import load_audio
from src.preprocess import bandpass_filter, pre_emphasis, energy_endpoint_detect

path = next(Path(config.RAW_DIR, "cough").glob("*.wav"))
y, sr = load_audio(path)
yb = bandpass_filter(y, sr=sr); yp = pre_emphasis(yb)
segs = energy_endpoint_detect(yp, sr=sr)
t = np.arange(len(y))/sr
fig, ax = plt.subplots(3,1, figsize=(12,7), sharex=True)
ax[0].plot(t, y, lw=.6); ax[0].set_title("原始波形"); ax[0].set_ylabel("幅度")
ax[1].plot(t, yp, lw=.6); ax[1].set_title("滤波+预加重后"); ax[1].set_ylabel("幅度")
ax[2].plot(t, yp, lw=.6, color="gray")
for s,e in segs:
    ax[2].axvspan(s/sr, e/sr, color="orange", alpha=.4)
ax[2].set_title(f"端点检测（橙色=有效段，共{len(segs)}段）"); ax[2].set_xlabel("时间/s")
plt.tight_layout(); plt.show()

## 3. 步骤2：特征提取（MFCC 39 维）

MFCC(13) + Δ(13) + Δ²(13)，每个样本 (101, 39) 特征矩阵。

In [ ]:
from src.preprocess import extract_voice_segments
from src.features import extract_feature_matrix

segs = extract_voice_segments(y, sr=sr)
feat = extract_feature_matrix(segs[0], sr=sr)
print("特征矩阵形状:", feat.shape)  # (101, 39)
plt.figure(figsize=(12,4))
plt.imshow(feat.T, aspect="auto", origin="lower", cmap="viridis")
plt.colorbar(); plt.xlabel("帧 (10ms)"); plt.ylabel("39 维特征")
plt.title("MFCC+Δ+Δ² 特征图（咳嗽段）"); plt.show()

## 4. 步骤3：构建数据集 + 8:1:1 划分

In [ ]:
get_ipython().system('python ../scripts/01_preprocess_audio.py')
get_ipython().system('python ../scripts/02_extract_features.py')
get_ipython().system('python ../scripts/03_build_dataset.py')
from src.dataset import load_split
X_train, y_train, X_val, y_val, X_test, y_test = load_split()
print("训练/验证/测试:", len(y_train), len(y_val), len(y_test))
print("训练集分布 咳嗽/非咳嗽:", int((y_train==0).sum()), "/", int((y_train==1).sum()))

## 5. 步骤4：训练 CNN-BiLSTM

> Notebook 演示用 30 轮（`config.EPOCHS` 默认 100，训练脚本用完整设置）。

In [ ]:
import config
config.EPOCHS = 30
from src.train import train_once
train_once(X_train, y_train, X_val, y_val, X_test, y_test,
           model_path=config.MODEL_DIR / "best_model.h5", tag="model")
from IPython.display import Image
Image(filename=str(config.RESULT_DIR / "history_model.png"))

## 6. 步骤5：评估（混淆矩阵）

In [ ]:
from src.inference import load_model
from src.model import add_channel
model = load_model()
y_pred = np.argmax(model.predict(add_channel(X_test), verbose=0), axis=1)
from src.evaluate import compute_metrics, plot_confusion_matrix
print(compute_metrics(y_test, y_pred))
plot_confusion_matrix(y_test, y_pred, classes=config.CLASS_NAMES, title="Test Confusion Matrix")
plt.show()

## 7. 步骤6：长音频滑窗检测 + 步骤7：监测预警

对示例音频做滑窗检测，再用监测器统计频次并给出预警级别。

In [ ]:
from src.inference import load_model, predict_long_audio
from src.monitor import CoughMonitor

model = load_model()
res = predict_long_audio(model, r"..\..\data\raw\six_class\app\app_1.wav")
print("检测到咳嗽事件:", res["cough_count"], "个 ->", res["events"])

monitor = CoughMonitor(pig_house_id="A栋3号舍")
monitor.record_audio(r"..\..\data\raw\six_class\app\app_1.wav")
import json
print(json.dumps(monitor.status_json(), ensure_ascii=False, indent=2))